In [1]:
%%writefile server_app.py
import os
from fastapi import FastAPI
from langserve import add_routes
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

app = FastAPI(
    title="Smart Contract Summary & Q&A Assistant Server",
    description="An AI-powered server for analyzing smart contracts and documents using RAG logic.",
    version="1.0.0"
)

llm = ChatOpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN"),
    model="meta-llama/Llama-3.1-8B-Instruct:novita",
    temperature=0.4, 
    max_tokens=1000
)

system_instruction = """
You are a professional Document Q&A Assistant.

RULES:
- Answer ONLY from the provided CONTEXT.
- Do NOT invent facts or page numbers.
- If the answer is not found, say:
  "I cannot find this information in the document."
- For greetings: reply politely and briefly.
- If asked who you are: say you are an AI Document Assistant that answers based on uploaded files.

CITATIONS:
- When giving facts, always add: "Source: Page X".
- If page is unclear, say: "Source page unclear."

OUTPUT:
Answer
Sources:

CONTEXT:
{context}
"""


prompt = ChatPromptTemplate.from_messages([
    ("system", system_instruction),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()
add_routes(app, chain, path="/rag_chat")

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=9017)

Overwriting server_app.py
